In [ ]:
# Cell 1: Install everything
# After this finishes, restart the session
# then skip this cell and run from Cell 2 onward

!pip install --upgrade transformers datasets peft bitsandbytes trl accelerate \
    evaluate rouge_score nltk scikit-learn matplotlib seaborn numpy==1.26.4

import torch
import transformers
import bitsandbytes

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"bitsandbytes: {bitsandbytes.__version__}")
print(f"transformers: {transformers.__version__}")

In [ ]:
# Cell 2: HuggingFace login + Mount Google Drive
from huggingface_hub import login
login(token="YOUR_HUGGINGFACE_TOKEN")

from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted!")

In [ ]:
# Cell 3: Load the Bitext dataset
from datasets import load_dataset

dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
df = dataset["train"].to_pandas()

print(f"Total examples: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"Unique intents: {df['intent'].nunique()}")
print(f"Unique categories: {df['category'].nunique()}")
print(f"\nSample row:")
print(df.iloc[0])

In [ ]:
# Cell 4: Quick EDA
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

intent_counts = df['intent'].value_counts()
axes[0].barh(intent_counts.index, intent_counts.values)
axes[0].set_title(f"Intent Distribution ({df['intent'].nunique()} intents)")
axes[0].set_xlabel("Count")

cat_counts = df['category'].value_counts()
axes[1].pie(cat_counts.values, labels=cat_counts.index, autopct='%1.1f%%')
axes[1].set_title(f"Category Distribution ({df['category'].nunique()} categories)")

plt.tight_layout()
plt.savefig("eda_distributions.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nIntent stats: mean={intent_counts.mean():.0f}, std={intent_counts.std():.0f}")
print(f"Min examples per intent: {intent_counts.min()}")
print(f"Max examples per intent: {intent_counts.max()}")

In [ ]:
# Cell 5: Define the formatting function

SYSTEM_PROMPT = """You are a helpful customer support assistant. You provide clear,
accurate, and professional responses to customer inquiries. Be concise but thorough,
and always maintain a polite and supportive tone."""

def format_single_turn(row):
    """Format a single example into Mistral instruct format."""
    instruction = row["instruction"]
    response = row["response"]
    intent = row["intent"]

    text = f"<s>[INST] {SYSTEM_PROMPT}\n\nCustomer: {instruction} [/INST] {response}</s>"

    return {
        "text": text,
        "intent": intent,
        "category": row["category"]
    }

In [ ]:
# Cell 6: Create multi-turn conversation pairs
import random
random.seed(42)

def create_multi_turn_examples(df, num_multi_turn=5000, max_turns=3):
    multi_turn_examples = []
    categories = df['category'].unique()

    for _ in range(num_multi_turn):
        cat = random.choice(categories)
        cat_df = df[df['category'] == cat]

        num_turns = random.randint(2, max_turns)
        if len(cat_df) < num_turns:
            continue

        sampled = cat_df.sample(n=num_turns, replace=False)

        conversation = f"<s>[INST] {SYSTEM_PROMPT}\n\n"

        for i, (_, row) in enumerate(sampled.iterrows()):
            if i < num_turns - 1:
                conversation += f"Customer: {row['instruction']} [/INST] {row['response']}</s>"
                conversation += f"<s>[INST] "
            else:
                conversation += f"Customer: {row['instruction']} [/INST] {row['response']}</s>"

        multi_turn_examples.append({
            "text": conversation,
            "intent": sampled.iloc[-1]['intent'],
            "category": cat
        })

    return multi_turn_examples

multi_turn_data = create_multi_turn_examples(df, num_multi_turn=5000)
print(f"Created {len(multi_turn_data)} multi-turn examples")
print(f"\nSample multi-turn conversation (first 500 chars):")
print(multi_turn_data[0]["text"][:500])

In [ ]:
# Cell 7: Combine and create train/val/test splits
from sklearn.model_selection import train_test_split
from datasets import Dataset

single_turn_data = [format_single_turn(row) for _, row in df.iterrows()]
print(f"Single-turn examples: {len(single_turn_data)}")
print(f"Multi-turn examples: {len(multi_turn_data)}")

all_data = single_turn_data + multi_turn_data
random.shuffle(all_data)
print(f"Total combined: {len(all_data)}")

intents = [d["intent"] for d in all_data]

train_data, temp_data, train_intents, temp_intents = train_test_split(
    all_data, intents, test_size=0.2, random_state=42, stratify=intents
)

val_data, test_data, _, _ = train_test_split(
    temp_data, temp_intents, test_size=0.5, random_state=42, stratify=temp_intents
)

print(f"\nTrain: {len(train_data)}")
print(f"Validation: {len(val_data)}")
print(f"Test: {len(test_data)}")

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)
test_dataset = Dataset.from_list(test_data)

print(f"\nDatasets created successfully!")
print(f"Sample training text (first 300 chars): {train_dataset[0]['text'][:300]}")

In [ ]:
# Cell 8: Check token lengths
from transformers import AutoTokenizer
import numpy as np

model_name = "mistralai/Mistral-7B-Instruct-v0.3"
tokenizer = AutoTokenizer.from_pretrained(model_name)

token_lengths = [len(tokenizer.encode(ex["text"])) for ex in train_data]

print(f"Token length stats:")
print(f"  Mean: {np.mean(token_lengths):.0f}")
print(f"  Median: {np.median(token_lengths):.0f}")
print(f"  Max: {np.max(token_lengths)}")
print(f"  95th percentile: {np.percentile(token_lengths, 95):.0f}")
print(f"  % under 512 tokens: {100 * np.mean(np.array(token_lengths) <= 512):.1f}%")
print(f"  % under 1024 tokens: {100 * np.mean(np.array(token_lengths) <= 1024):.1f}%")

In [ ]:
# Cell 9: Load Mistral-7B in 4-bit quantization
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# 4-bit quant config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# Mistral has no pad token by default
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model.config.use_cache = False  # Required for gradient checkpointing

print(f"Model loaded! Memory usage: {model.get_memory_footprint() / 1e9:.2f} GB")

In [ ]:
# Cell 10: Configure LoRA adapters
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Force all parameters to float16 so the grad scaler doesn't crash on T4
for name, param in model.named_parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)
print("All parameters verified as float16")

In [ ]:
# Cell 11+12: Configure and START TRAINING
# Saves directly to Google Drive so checkpoints survive runtime disconnects
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "PATH_TO_OUTPUT_DIR"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    max_steps=750,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=150,
    save_strategy="steps",
    save_steps=150,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    # no mixed precision flags here
    # model is already float16 via torch_dtype
    # fp16=True adds a GradScaler that breaks bitsandbytes
    # bf16=True isn't supported on T4
    fp16=False,
    bf16=False,
    report_to="none",
    seed=42,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataset_text_field="text",
    max_length=512,
    packing=True,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

print("Starting training...")
print(f"Train examples: {len(train_dataset)}")
print(f"Eval examples: {len(val_dataset)}")
print(f"Max steps: 750")
print(f"Effective batch size: {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps}")
print(f"Saving checkpoints to Google Drive: {OUTPUT_DIR}")

train_result = trainer.train()

trainer.save_model(f"{OUTPUT_DIR}/final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final")

print(f"\nTraining complete!")
print(f"Final train loss: {train_result.training_loss:.4f}")

In [ ]:
# Cell 13: Plot training curves
import matplotlib.pyplot as plt

logs = trainer.state.log_history

train_losses = [(l["step"], l["loss"]) for l in logs if "loss" in l and "eval_loss" not in l]
eval_losses = [(l["step"], l["eval_loss"]) for l in logs if "eval_loss" in l]

fig, ax = plt.subplots(figsize=(10, 5))
if train_losses:
    steps, losses = zip(*train_losses)
    ax.plot(steps, losses, label="Train Loss", alpha=0.7)
if eval_losses:
    steps, losses = zip(*eval_losses)
    ax.plot(steps, losses, label="Eval Loss", linewidth=2)

ax.set_xlabel("Steps")
ax.set_ylabel("Loss")
ax.set_title("Training & Validation Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.savefig("PATH_TO_OUTPUT_DIR/training_curves.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 18: Zero-shot baseline comparison
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)
base_tokenizer = AutoTokenizer.from_pretrained(model_name)
base_tokenizer.pad_token = base_tokenizer.eos_token
base_model.eval()

baseline_results = []
for idx in tqdm(eval_indices[:100], desc="Baseline generation"):
    example = test_dataset[idx]
    text = example["text"]

    last_inst_start = text.rfind("Customer: ") + len("Customer: ")
    last_inst_end = text.rfind(" [/INST]")
    instruction = text[last_inst_start:last_inst_end]

    gt_start = text.rfind("[/INST] ") + len("[/INST] ")
    gt_end = text.rfind("</s>")
    ground_truth = text[gt_start:gt_end]

    prompt = f"<s>[INST] {SYSTEM_PROMPT}\n\nCustomer: {instruction} [/INST]"
    inputs = base_tokenizer(prompt, return_tensors="pt").to(base_model.device)

    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=base_tokenizer.eos_token_id,
        )

    prediction = base_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

    baseline_results.append({
        "instruction": instruction,
        "ground_truth": ground_truth,
        "prediction": prediction,
        "intent": example["intent"],
    })

baseline_preds = [r["prediction"] for r in baseline_results]
baseline_refs = [r["ground_truth"] for r in baseline_results]

baseline_rouge = rouge.compute(predictions=baseline_preds, references=baseline_refs)

ft_preds_100 = [r["prediction"] for r in results[:100]]
ft_refs_100 = [r["ground_truth"] for r in results[:100]]
ft_rouge_100 = rouge.compute(predictions=ft_preds_100, references=ft_refs_100)

print("=" * 60)
print("COMPARISON: Fine-tuned QLoRA vs Zero-shot Baseline")
print("=" * 60)
print(f"{'Metric':<15} {'Fine-tuned':<15} {'Zero-shot':<15} {'Improvement':<15}")
print("-" * 60)
for key in baseline_rouge:
    ft_val = ft_rouge_100[key]
    bl_val = baseline_rouge[key]
    improvement = ((ft_val - bl_val) / bl_val * 100) if bl_val > 0 else 0
    print(f"{key:<15} {ft_val:<15.4f} {bl_val:<15.4f} {improvement:+.1f}%")

In [ ]:
# PLOT: Ablation Comparison Chart

import matplotlib.pyplot as plt
ranks = sorted(ablation_results.keys())
train_losses_abl = [ablation_results[r]["train_loss"] for r in ranks]
eval_losses_abl = [ablation_results[r]["eval_loss"] for r in ranks]

fig, ax = plt.subplots(figsize=(8, 5))
x = range(len(ranks))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], train_losses_abl, width, label='Train Loss', color='#7c3aed', alpha=0.8)
bars2 = ax.bar([i + width/2 for i in x], eval_losses_abl, width, label='Val Loss', color='#a855f7', alpha=0.8)

ax.set_xlabel('LoRA Rank')
ax.set_ylabel('Loss')
ax.set_title('Ablation Study: Effect of LoRA Rank on Performance')
ax.set_xticks(x)
ax.set_xticklabels([f'Rank {r}' for r in ranks])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005, f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005, f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig("PATH_TO_OUTPUT_DIR/ablation_study.png", dpi=150, bbox_inches='tight')
plt.show()

print("Ablation chart saved to Drive!")

In [ ]:
# Cell 15: Set up inference
from peft import PeftModel

# if continuing from training the model is already loaded
# if Colab reconnected, uncomment below

model = AutoModelForCausalLM.from_pretrained(
     model_name,
     quantization_config=bnb_config,
     torch_dtype=torch.float16,
     device_map="auto",
 )
model = PeftModel.from_pretrained(model, "PATH_TO_ADAPTER")
tokenizer = AutoTokenizer.from_pretrained("PATH_TO_ADAPTER")

model.eval()

def generate_response(instruction, history=None, max_new_tokens=256):
    """Generate a response from the fine-tuned model."""
    if history:
        prompt = f"<s>[INST] {SYSTEM_PROMPT}\n\n"
        for h_user, h_assistant in history:
            prompt += f"Customer: {h_user} [/INST] {h_assistant}</s><s>[INST] "
        prompt += f"Customer: {instruction} [/INST]"
    else:
        prompt = f"<s>[INST] {SYSTEM_PROMPT}\n\nCustomer: {instruction} [/INST]"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip()

# Quick test
test_response = generate_response("I need to cancel my subscription")
print(f"Test response: {test_response}")

In [ ]:
!pip install -q sentence-transformers

In [ ]:
# METRIC 1: Latency Measurement

from tqdm import tqdm
import time

print("Measuring response latency...")
latency_samples = 50
latency_indices = random.sample(range(len(test_dataset)), latency_samples)
latencies = []

for idx in tqdm(latency_indices, desc="Latency test"):
    text = test_dataset[idx]["text"]
    last_inst_start = text.rfind("Customer: ") + len("Customer: ")
    last_inst_end = text.rfind(" [/INST]")
    instruction = text[last_inst_start:last_inst_end]

    start = time.time()
    response = generate_response(instruction)
    elapsed = (time.time() - start) * 1000  # milliseconds
    latencies.append(elapsed)

print(f"\n{'='*50}")
print(f"LATENCY RESULTS ({latency_samples} samples)")
print(f"{'='*50}")
print(f"Mean:   {np.mean(latencies):.0f} ms")
print(f"Median: {np.median(latencies):.0f} ms")
print(f"P95:    {np.percentile(latencies, 95):.0f} ms")
print(f"Min:    {np.min(latencies):.0f} ms")
print(f"Max:    {np.max(latencies):.0f} ms")

In [ ]:
# METRIC 2: Intent Classification F1 Score
# check if response matches the expected intent by keyword matching

from sklearn.metrics import classification_report, f1_score

# keywords that signal each intent
intent_keywords = {
    "cancel_order": ["cancel", "cancellation", "cancelled"],
    "change_order": ["change", "modify", "update your order", "edit order"],
    "change_shipping_address": ["shipping address", "delivery address", "address"],
    "check_cancellation_fee": ["cancellation fee", "charge", "penalty"],
    "check_invoice": ["invoice", "bill", "statement"],
    "check_payment_methods": ["payment method", "pay with", "accepted payment"],
    "check_refund_policy": ["refund policy", "money back", "return policy"],
    "complaint": ["sorry", "apologize", "understand your frustration", "complaint"],
    "contact_customer_service": ["contact", "reach out", "customer service", "support team"],
    "contact_human_agent": ["human agent", "real person", "speak to someone", "transfer"],
    "create_account": ["create", "sign up", "register", "new account"],
    "delete_account": ["delete", "remove", "close your account", "deactivate"],
    "delivery_options": ["delivery option", "shipping method", "shipping option"],
    "delivery_period": ["delivery time", "how long", "estimated delivery", "business days"],
    "edit_account": ["edit", "update your account", "profile", "account details"],
    "get_invoice": ["invoice", "download", "send you", "email"],
    "get_refund": ["refund", "money back", "reimburse", "credited"],
    "newsletter_subscription": ["newsletter", "subscribe", "subscription", "unsubscribe"],
    "payment_issue": ["payment issue", "payment problem", "transaction", "declined"],
    "place_order": ["place order", "order", "purchase", "buy"],
    "recover_password": ["password", "reset", "recover", "forgot"],
    "registration_problems": ["registration", "sign up issue", "cannot register"],
    "review": ["review", "feedback", "rating"],
    "set_up_shipping_address": ["set up", "add address", "shipping address"],
    "switch_account": ["switch", "upgrade", "premium", "account type"],
    "track_order": ["track", "tracking", "status", "where is"],
    "track_refund": ["track refund", "refund status", "where is my refund"],
}

def classify_response(response_text, intent_keywords):
    """Classify a response by matching keywords to intents."""
    response_lower = response_text.lower()
    scores = {}
    for intent, keywords in intent_keywords.items():
        score = sum(1 for kw in keywords if kw in response_lower)
        if score > 0:
            scores[intent] = score
    if scores:
        return max(scores, key=scores.get)
    return "unknown"

# Generate and classify 200 test examples
print("Computing F1 scores...")
f1_samples = 200
f1_indices = random.sample(range(len(test_dataset)), f1_samples)

true_intents = []
pred_intents = []

for idx in tqdm(f1_indices, desc="F1 evaluation"):
    example = test_dataset[idx]
    text = example["text"]
    true_intent = example["intent"]

    last_inst_start = text.rfind("Customer: ") + len("Customer: ")
    last_inst_end = text.rfind(" [/INST]")
    instruction = text[last_inst_start:last_inst_end]

    response = generate_response(instruction)
    pred_intent = classify_response(response, intent_keywords)

    true_intents.append(true_intent)
    pred_intents.append(pred_intent)

# Compute F1
all_intents = list(intent_keywords.keys())
macro_f1 = f1_score(true_intents, pred_intents, labels=all_intents, average="macro", zero_division=0)
weighted_f1 = f1_score(true_intents, pred_intents, labels=all_intents, average="weighted", zero_division=0)

print(f"\n{'='*50}")
print(f"INTENT CLASSIFICATION F1 ({f1_samples} samples)")
print(f"{'='*50}")
print(f"Macro F1:    {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")

# Accuracy
correct = sum(1 for t, p in zip(true_intents, pred_intents) if t == p)
print(f"Accuracy:    {correct}/{f1_samples} ({100*correct/f1_samples:.1f}%)")

In [ ]:
# METRIC 3: Ablation Study, LoRA Rank 8 vs 16 vs 32
# train short 200-step runs at different ranks and compare loss

from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer, SFTConfig
from trl import SFTTrainer, SFTConfig

ablation_results = {}

for rank in [8, 32]:  # rank 16 is our main run, we already have those
    print(f"\n{'='*50}")
    print(f"ABLATION: Training with LoRA rank={rank}")
    print(f"{'='*50}")

    # Load fresh base model
    abl_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    abl_model = prepare_model_for_kbit_training(abl_model)

    abl_lora_config = LoraConfig(
        r=rank,
        lora_alpha=rank * 2,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )

    abl_model = get_peft_model(abl_model, abl_lora_config)
    abl_model.print_trainable_parameters()

    # Force float16
    for name, param in abl_model.named_parameters():
        if param.dtype == torch.bfloat16:
            param.data = param.data.to(torch.float16)

    abl_tokenizer = AutoTokenizer.from_pretrained(model_name)
    abl_tokenizer.pad_token = abl_tokenizer.eos_token

    output_dir = f"/content/ablation_rank_{rank}"

    abl_config = SFTConfig(
        output_dir=output_dir,
        num_train_epochs=1,
        max_steps=200,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=5e-5,
        weight_decay=0.01,
        warmup_steps=20,
        lr_scheduler_type="cosine",
        logging_steps=25,
        eval_strategy="steps",
        eval_steps=100,
        fp16=False,
        bf16=False,
        report_to="none",
        seed=42,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        dataset_text_field="text",
        max_length=512,
        packing=True,
    )

    abl_trainer = SFTTrainer(
        model=abl_model,
        args=abl_config,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=abl_tokenizer,
    )

    abl_result = abl_trainer.train()

    # Get eval loss
    eval_results = abl_trainer.evaluate()
    eval_loss = eval_results["eval_loss"]

    # Get training losses from log
    train_losses = [(l["step"], l["loss"]) for l in abl_trainer.state.log_history if "loss" in l and "eval_loss" not in l]
    final_train_loss = train_losses[-1][1] if train_losses else None

    trainable = sum(p.numel() for p in abl_model.parameters() if p.requires_grad)

    ablation_results[rank] = {
        "train_loss": final_train_loss,
        "eval_loss": eval_loss,
        "trainable_params": trainable,
        "train_losses": train_losses,
    }

    print(f"Rank {rank}: Train Loss={final_train_loss:.4f}, Eval Loss={eval_loss:.4f}, Params={trainable:,}")

    # Clean up GPU memory
    del abl_model, abl_trainer
    torch.cuda.empty_cache()

# Add rank 16 results from our main training
ablation_results[16] = {
    "train_loss": 0.813,  # step 150 from main run
    "eval_loss": 0.776,   # best val loss from main run
    "trainable_params": 13_631_488,
}

print(f"\n\n{'='*60}")
print(f"ABLATION STUDY RESULTS")
print(f"{'='*60}")
print(f"{'Rank':<8} {'Params':<15} {'Train Loss':<15} {'Val Loss':<15}")
print("-" * 53)
for rank in sorted(ablation_results.keys()):
    r = ablation_results[rank]
    print(f"{rank:<8} {r['trainable_params']:>12,}   {r['train_loss']:<15.4f} {r['eval_loss']:<15.4f}")

In [ ]:
# Fix: Use logged eval loss instead of separate evaluate() call
# Rank 8 results from the training log above
ablation_results = {}
ablation_results[8] = {
    "train_loss": 0.749191,
    "eval_loss": 0.761840,
    "trainable_params": sum(p.numel() for p in abl_model.parameters() if p.requires_grad),
}
print(f"Rank 8 saved: {ablation_results[8]}")

# Clean up rank 8
del abl_model, abl_trainer
torch.cuda.empty_cache()

# Now train rank 32
print(f"\n{'='*50}")
print(f"ABLATION: Training with LoRA rank=32")
print(f"{'='*50}")

abl_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)
abl_model = prepare_model_for_kbit_training(abl_model)

abl_lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

abl_model = get_peft_model(abl_model, abl_lora_config)
abl_model.print_trainable_parameters()

for name, param in abl_model.named_parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)

abl_tokenizer = AutoTokenizer.from_pretrained(model_name)
abl_tokenizer.pad_token = abl_tokenizer.eos_token

abl_config = SFTConfig(
    output_dir="/content/ablation_rank_32",
    num_train_epochs=1,
    max_steps=200,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=20,
    lr_scheduler_type="cosine",
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    fp16=False,
    bf16=False,
    report_to="none",
    seed=42,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataset_text_field="text",
    max_length=512,
    packing=True,
)

abl_trainer = SFTTrainer(
    model=abl_model,
    args=abl_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=abl_tokenizer,
)

abl_result = abl_trainer.train()

# Get eval loss from training logs instead of calling evaluate()
eval_losses = [(l["step"], l["eval_loss"]) for l in abl_trainer.state.log_history if "eval_loss" in l]
train_losses = [(l["step"], l["loss"]) for l in abl_trainer.state.log_history if "loss" in l and "eval_loss" not in l]

ablation_results[32] = {
    "train_loss": train_losses[-1][1],
    "eval_loss": eval_losses[-1][1],
    "trainable_params": sum(p.numel() for p in abl_model.parameters() if p.requires_grad),
}

# Add rank 16 from main training
ablation_results[16] = {
    "train_loss": 0.813,
    "eval_loss": 0.776,
    "trainable_params": 13_631_488,
}

print(f"\n\n{'='*60}")
print(f"ABLATION STUDY RESULTS")
print(f"{'='*60}")
print(f"{'Rank':<8} {'Params':<15} {'Train Loss':<15} {'Val Loss':<15}")
print("-" * 53)
for rank in sorted(ablation_results.keys()):
    r = ablation_results[rank]
    print(f"{rank:<8} {r['trainable_params']:>12,}   {r['train_loss']:<15.4f} {r['eval_loss']:<15.4f}")

In [ ]:
# SAVE ALL NEW METRICS TO DRIVE
import json

new_metrics = {
    "latency": {
        "mean_ms": round(np.mean(latencies), 0),
        "median_ms": round(np.median(latencies), 0),
        "p95_ms": round(np.percentile(latencies, 95), 0),
        "min_ms": round(np.min(latencies), 0),
        "max_ms": round(np.max(latencies), 0),
    },
    "intent_f1": {
        "macro_f1": round(macro_f1, 4),
        "weighted_f1": round(weighted_f1, 4),
        "accuracy": round(correct / f1_samples, 4),
    },
    "ablation": {
        str(r): {"train_loss": round(v["train_loss"], 4), "eval_loss": round(v["eval_loss"], 4), "params": v["trainable_params"]}
        for r, v in ablation_results.items()
    },
}

with open("PATH_TO_OUTPUT_DIR/additional_metrics.json", "w") as f:
    json.dump(new_metrics, f, indent=2)

print("All metrics saved to Drive!")
print(json.dumps(new_metrics, indent=2))

In [ ]:
# Cell: Compute Perplexity on Test Set
import torch
import numpy as np
from tqdm import tqdm

def compute_perplexity(model, tokenizer, dataset, num_samples=200):
    """Compute perplexity on a subset of the test set."""
    model.eval()
    total_loss = 0
    total_tokens = 0

    indices = random.sample(range(len(dataset)), min(num_samples, len(dataset)))

    for idx in tqdm(indices, desc="Computing perplexity"):
        text = dataset[idx]["text"]
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
            loss = outputs.loss

        num_tokens = inputs["input_ids"].shape[1]
        total_loss += loss.item() * num_tokens
        total_tokens += num_tokens

    avg_loss = total_loss / total_tokens
    perplexity = np.exp(avg_loss)
    return perplexity, avg_loss

# Fine-tuned model perplexity
print("Computing fine-tuned model perplexity...")
ft_perplexity, ft_loss = compute_perplexity(model, tokenizer, test_dataset, num_samples=200)
print(f"Fine-tuned Model. Perplexity: {ft_perplexity:.2f}, Avg Loss: {ft_loss:.4f}")

# Baseline perplexity (load base model without adapter)
print("\nLoading base model for baseline perplexity...")
from transformers import AutoModelForCausalLM
base_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3",
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)
base_tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")
base_tokenizer.pad_token = base_tokenizer.eos_token
base_model.eval()

print("Computing baseline perplexity...")
bl_perplexity, bl_loss = compute_perplexity(base_model, base_tokenizer, test_dataset, num_samples=200)
print(f"Zero-shot Baseline. Perplexity: {bl_perplexity:.2f}, Avg Loss: {bl_loss:.4f}")

# Summary
print(f"\n{'='*50}")
print(f"PERPLEXITY COMPARISON")
print(f"{'='*50}")
print(f"Fine-tuned:  {ft_perplexity:.2f}")
print(f"Zero-shot:   {bl_perplexity:.2f}")
improvement = ((bl_perplexity - ft_perplexity) / bl_perplexity) * 100
print(f"Improvement: {improvement:.1f}% lower (lower is better)")

In [ ]:
# Cell 16: Generate predictions on test set
from tqdm import tqdm

eval_subset_size = 500
eval_indices = random.sample(range(len(test_dataset)), min(eval_subset_size, len(test_dataset)))

results = []
for idx in tqdm(eval_indices, desc="Generating responses"):
    example = test_dataset[idx]

    text = example["text"]
    last_inst_start = text.rfind("Customer: ") + len("Customer: ")
    last_inst_end = text.rfind(" [/INST]")
    instruction = text[last_inst_start:last_inst_end]

    gt_start = text.rfind("[/INST] ") + len("[/INST] ")
    gt_end = text.rfind("</s>")
    ground_truth = text[gt_start:gt_end]

    prediction = generate_response(instruction)

    results.append({
        "instruction": instruction,
        "ground_truth": ground_truth,
        "prediction": prediction,
        "intent": example["intent"],
        "category": example["category"],
    })

print(f"\nGenerated {len(results)} predictions")

In [ ]:
# Cell 17: Compute ROUGE, BLEU scores
import evaluate
import numpy as np

rouge = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")

predictions = [r["prediction"] for r in results]
references = [r["ground_truth"] for r in results]

rouge_results = rouge.compute(predictions=predictions, references=references)
print("=== ROUGE Scores ===")
for key, value in rouge_results.items():
    print(f"  {key}: {value:.4f}")

bleu_scores = []
for pred, ref in zip(predictions, references):
    try:
        score = bleu_metric.compute(
            predictions=[pred],
            references=[[ref]],
        )
        bleu_scores.append(score["bleu"])
    except:
        bleu_scores.append(0.0)

print(f"\n=== BLEU Score ===")
print(f"  Average BLEU: {np.mean(bleu_scores):.4f}")
print(f"  Median BLEU: {np.median(bleu_scores):.4f}")

In [ ]:
# Cell 19: Qualitative analysis, good and bad examples
from rouge_score import rouge_scorer

print("=" * 60)
print("QUALITATIVE ANALYSIS")
print("=" * 60)

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

for r in results:
    score = scorer.score(r["ground_truth"], r["prediction"])
    r["rougeL"] = score["rougeL"].fmeasure

sorted_results = sorted(results, key=lambda x: x["rougeL"], reverse=True)

print("\n--- TOP 5 BEST RESPONSES ---")
for i, r in enumerate(sorted_results[:5]):
    print(f"\n[{i+1}] Intent: {r['intent']} | ROUGE-L: {r['rougeL']:.3f}")
    print(f"  Customer: {r['instruction'][:100]}...")
    print(f"  Expected: {r['ground_truth'][:150]}...")
    print(f"  Generated: {r['prediction'][:150]}...")

print("\n\n--- TOP 5 WORST RESPONSES ---")
for i, r in enumerate(sorted_results[-5:]):
    print(f"\n[{i+1}] Intent: {r['intent']} | ROUGE-L: {r['rougeL']:.3f}")
    print(f"  Customer: {r['instruction'][:100]}...")
    print(f"  Expected: {r['ground_truth'][:150]}...")
    print(f"  Generated: {r['prediction'][:150]}...")

In [ ]:
# Cell 20: Per-intent performance breakdown
import pandas as pd
import matplotlib.pyplot as plt

intent_scores = {}
for r in results:
    intent = r["intent"]
    if intent not in intent_scores:
        intent_scores[intent] = []
    intent_scores[intent].append(r["rougeL"])

intent_df = pd.DataFrame([
    {"Intent": k, "Mean ROUGE-L": np.mean(v), "Count": len(v)}
    for k, v in intent_scores.items()
]).sort_values("Mean ROUGE-L", ascending=False)

print("Per-Intent Performance:")
print(intent_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(intent_df["Intent"], intent_df["Mean ROUGE-L"])
ax.set_xlabel("Mean ROUGE-L Score")
ax.set_title("Performance by Intent Category")
ax.axvline(x=intent_df["Mean ROUGE-L"].mean(), color='red', linestyle='--', label='Average')
ax.legend()
plt.tight_layout()
plt.savefig("PATH_TO_OUTPUT_DIR/per_intent_performance.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 21: Multi-turn conversation demo
print("=" * 60)
print("MULTI-TURN CONVERSATION DEMO")
print("=" * 60)

history = []

queries = [
    "I need to cancel my subscription",
    "Can I get a refund for the remaining period?",
    "How long will the refund take to process?",
]

for query in queries:
    response = generate_response(query, history=history if history else None)
    print(f"\nCustomer: {query}")
    print(f"Assistant: {response}")
    history.append((query, response))

In [ ]:
# Cell 22: Multi-turn context retention
# checks if the model keeps entity references across 3 turns

import random
import numpy as np
from rouge_score import rouge_scorer

random.seed(42)

# turn 1 introduces an entity, turn 2 is unrelated, turn 3 refers back to it
multiturn_templates = [
    {
        "intent": "track_order",
        "turn1": "i need to track my order #{entity}",
        "turn2": "how long does shipping usually take?",
        "turn3": "and when will order #{entity} arrive specifically?",
        "gold": "I can help you track order #{entity}. Let me pull up the latest shipping status for your order #{entity} right now.",
    },
    {
        "intent": "cancel_order",
        "turn1": "i want to cancel order #{entity}",
        "turn2": "what's your cancellation policy?",
        "turn3": "ok cancel #{entity} then",
        "gold": "I'll cancel order #{entity} for you right away. You should receive a confirmation email once order #{entity} has been cancelled.",
    },
    {
        "intent": "get_refund",
        "turn1": "i need a refund for order #{entity}",
        "turn2": "how long do refunds take?",
        "turn3": "please process the refund for #{entity}",
        "gold": "I'll process the refund for order #{entity} now. The refund for #{entity} will appear in your account within 5-10 business days.",
    },
    {
        "intent": "change_shipping_address",
        "turn1": "i need to change the shipping address for order #{entity}",
        "turn2": "do you ship internationally?",
        "turn3": "can you update the address on order #{entity}?",
        "gold": "I can update the shipping address on order #{entity}. Please provide the new address and I will apply it to order #{entity}.",
    },
    {
        "intent": "check_invoice",
        "turn1": "where can i find invoice #{entity}?",
        "turn2": "what payment methods do you accept?",
        "turn3": "i still need to see invoice #{entity}",
        "gold": "You can find invoice #{entity} in your account under Billing History. I will also email a copy of invoice #{entity} to your registered address.",
    },
]

# build 50 convos, 10 per template
multiturn_test = []
for i in range(50):
    template = multiturn_templates[i % len(multiturn_templates)]
    entity_val = str(random.randint(10000, 99999))
    multiturn_test.append({
        "intent": template["intent"],
        "entity_value": entity_val,
        "turn1": template["turn1"].format(entity=entity_val),
        "turn2": template["turn2"],
        "turn3": template["turn3"].format(entity=entity_val),
        "gold": template["gold"].format(entity=entity_val),
    })

print(f"Generated {len(multiturn_test)} three-turn test conversations")
print(f"Example: {multiturn_test[0]}")
print()

# run all 50 through the fine-tuned model
print("Running fine-tuned model on 50 multi-turn conversations...")
print()

ft_results = []
for i, convo in enumerate(multiturn_test):
    # turn 1
    r1 = generate_response(convo["turn1"], history=None)
    # turn 2 with turn 1 in history
    r2 = generate_response(convo["turn2"], history=[(convo["turn1"], r1)])
    # turn 3 with full history, this one gets scored
    r3 = generate_response(convo["turn3"], history=[(convo["turn1"], r1), (convo["turn2"], r2)])

    ft_results.append({**convo, "r1": r1, "r2": r2, "r3": r3})
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/50 done")

print()
print("Computing ROUGE-L scores and entity preservation...")

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
ft_rouge_l_scores = []
entity_preserved_count = 0
failures = []

for r in ft_results:
    score = scorer.score(r["gold"], r["r3"])['rougeL'].fmeasure
    ft_rouge_l_scores.append(score)
    if r["entity_value"] in r["r3"]:
        entity_preserved_count += 1
    else:
        failures.append({
            "intent": r["intent"],
            "entity": r["entity_value"],
            "turn3_response": r["r3"][:200],
        })

mean_rouge_l = float(np.mean(ft_rouge_l_scores))

print()
print("=" * 60)
print("MULTI-TURN EVALUATION RESULTS")
print("=" * 60)
print(f"ROUGE-L on turn 3 (n=50):       {mean_rouge_l:.4f}")
print(f"Entity preserved across turns:  {entity_preserved_count}/50")
print(f"Entity preservation rate:       {entity_preserved_count/50*100:.1f}%")
print("=" * 60)
print()
print(f"Showing first 3 failure cases (entity not preserved in turn 3):")
for f in failures[:3]:
    print(f"  Intent: {f['intent']}, Entity #{f['entity']} missing from response")
    print(f"  Response start: {f['turn3_response'][:120]}...")
    print()

# save to Drive
import json as json_lib
multiturn_save = {
    "n_conversations": 50,
    "mean_rouge_l_turn3": mean_rouge_l,
    "entity_preserved": entity_preserved_count,
    "entity_preservation_rate": entity_preserved_count / 50,
    "individual_scores": ft_rouge_l_scores,
    "results": ft_results,
}
with open("PATH_TO_OUTPUT_DIR/multiturn_results.json", "w") as f:
    json_lib.dump(multiturn_save, f, indent=2)

print("Multi-turn results saved to Drive.")


In [ ]:
import json

save_data = {
    "results": results,
    "rouge_results": rouge_results,
    "bleu_scores": bleu_scores,
    "baseline_rouge": baseline_rouge,
    "ft_rouge_100": ft_rouge_100,
}

with open("PATH_TO_OUTPUT_DIR/eval_results.json", "w") as f:
    json.dump(save_data, f)

print("Everything saved to Drive!")